In [6]:
# Core data manipulation and numerical computing
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning -- preprocessing & model selection
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

# Machine Learning -- classifiers
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb

# Imbalanced-learn -- resampling
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

# Evaluation metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score,
    precision_recall_fscore_support, balanced_accuracy_score
)

# Clustering (used in V-feature engineering)
from sklearn.cluster import KMeans

# Global settings
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('All libraries loaded successfully.')
print(f'Random seed: {RANDOM_STATE}')

All libraries loaded successfully.
Random seed: 42


In [ ]:
df_preprocess = pd.read_pickle('ieee_fraud_featured.pkl')
df_preprocess.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,d_features_mean,d_features_std,amount_hour_interaction,amount_weekend_interaction,addr1_fraud_rate,addr1_count,addr2_fraud_rate,addr2_count,device_fraud_rate,device_info_length
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,10.600000,5.941380,0.000000,0.0,0.017809,23078.0,0.023972,520481.0,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,0.000000,0.000000,0.008056,0.0,0.025426,42751.0,0.023972,520481.0,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,126.000000,172.532606,1.130833,0.0,0.031955,26287.0,0.023972,520481.0,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,73.285714,51.162207,1.375000,0.0,0.031652,9478.0,0.023972,520481.0,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.000000,NaN,1.472222,0.0,0.032672,3581.0,0.023972,520481.0,0.0,29.0


In [4]:
TARGET = 'isFraud'
EXCLUDE = [TARGET, 'TransactionID']
feature_cols = [cols for cols in df_preprocess.columns if cols not in EXCLUDE]

X = df_preprocess[feature_cols].copy()
y = df_preprocess[TARGET].copy()

print(f'Features: {X.shape[1]}')
print(f'Samples: {y.shape[0]}')
print(f'Fraud rate: {y.mean():.3%}')

Features: 507
Samples: 590540
Fraud rate: 3.499%


In [7]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f'Categorical columns to encode:{len(cat_cols)}')

label_encoder = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoder[col] = le

    print('Encoding complete.')

Categorical columns to encode:32
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.
Encoding complete.


In [8]:
#Drop data with > 95% missing value

missing_pct_X = (X.isnull().sum() / len(X)) * 100
high_missing = missing_pct_X[missing_pct_X > 95].index.tolist()
X = X.drop(columns=high_missing)
print(f'Dropped {len(high_missing)} features with > 95% missing value')

#Imputation (replace missing value using statistical method)
num_cols = X.select_dtypes(include=[np.number]).columns.to_list()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.to_list()

if num_cols:
    num_imputer = SimpleImputer(strategy='median')
    X[num_cols] = num_imputer.fit_transform(X[num_cols])
    print(f'Imputed {len(num_cols)} numerical features with median')

if cat_cols:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
    print(f'Imputed {len(cat_cols)} categorical feature with mode')

Dropped 7 features with > 95% missing value
Imputed 500 numerical features with median
